# 01 · Upload training data

Custom-classifier training reads its samples **from Azure Blob Storage**, so this
notebook:

1. Reuses (or creates) a storage account and a `classifier-training` container.
2. Grants the **uploader** identity write access and uploads the two classes as
   blob prefixes - `invoice/` and `claim/`.
3. Grants the Foundry resource's managed identity storage access for optional
   managed-identity/private-network configurations. Notebook `02` uses the
   container-SAS flow documented by Microsoft Learn.

**Prereqs:** `az login` as an identity that can create the account + assign roles
(Owner / User Access Administrator on the resource group is enough).

## 1. Setup

Import the shared helpers and confirm the sample data is where we expect.

In [6]:
import os, sys
from pathlib import Path

# Make _lib importable regardless of the kernel's working directory.
_here = Path.cwd()
for _c in (_here, _here / 'notebooks', *_here.parents):
    if (_c / '_lib.py').exists():
        sys.path.insert(0, str(_c))
        break
import _lib

print('repo root :', _lib.REPO_ROOT)
print('invoices  :', _lib.INVOICES_DIR, '->', len(list(_lib.INVOICES_DIR.glob('*.pdf'))), 'pdf')
print('claims    :', _lib.CLAIMS_DIR, '->', len(list(_lib.CLAIMS_DIR.glob('*.pdf'))), 'pdf')
print('container :', _lib.CONTAINER)
print('classifier:', _lib.CLASSIFIER_ID)

repo root : C:\Users\jomedin\Documents\azure-ai-foundry-insurance
invoices  : C:\Users\jomedin\Documents\azure-ai-foundry-insurance\data\invoices -> 5 pdf
claims    : C:\Users\jomedin\Documents\azure-ai-foundry-insurance\data\propertyclaims -> 8 pdf
container : classifier-training
classifier: insurance-invoice-claim-v1


## 2. Storage account

Set `STORAGE_ACCOUNT_NAME` in `demo/classifier/.env` (3–24 lowercase alphanumeric
chars). The next cell defines a tiny `az` helper and reads your Azure context.

In [7]:
import subprocess, json

SUBSCRIPTION = os.environ.get('AZURE_SUBSCRIPTION_ID', '')
RESOURCE_GROUP = os.environ.get('AZURE_RESOURCE_GROUP', '')
ACCOUNT = os.environ.get('STORAGE_ACCOUNT_NAME', '')

def az(cmd_str, quiet=False):
    proc = subprocess.run(f'az {cmd_str}', capture_output=True, text=True, shell=True)
    if proc.returncode != 0:
        if not quiet:
            print('[az error]', (proc.stderr or proc.stdout).strip())
        return None
    out = proc.stdout.strip()
    try:
        return json.loads(out) if out else None
    except json.JSONDecodeError:
        return out

print('subscription  :', SUBSCRIPTION or '(unset)')
print('resource group:', RESOURCE_GROUP or '(unset)')
print('account name  :', ACCOUNT or '(unset - set STORAGE_ACCOUNT_NAME in .env)')

subscription  : 5784b6a5-de3f-4fa4-8b8f-e5bb70ff6b25
resource group: rg-insurance-workshop-dev-01
account name  : staidemosclsfrdev01


In [8]:
# Reuse ACCOUNT if it exists; otherwise create it (StorageV2, no public blob access).
if not ACCOUNT:
    raise SystemExit('Set STORAGE_ACCOUNT_NAME in demo/classifier/.env first.')

existing = az(f'storage account show -n {ACCOUNT} -g {RESOURCE_GROUP} --query id -o tsv', quiet=True)
if existing:
    print('reusing existing storage account:', ACCOUNT)
else:
    location = az(f'group show -n {RESOURCE_GROUP} --query location -o tsv') or 'eastus'
    print(f'creating storage account {ACCOUNT} in {location} ...')
    az(f'storage account create -n {ACCOUNT} -g {RESOURCE_GROUP} -l {location} '
       f'--sku Standard_LRS --kind StorageV2 --min-tls-version TLS1_2 --allow-blob-public-access false')

# Make the resolved name visible to the _lib helpers.
os.environ['STORAGE_ACCOUNT_NAME'] = ACCOUNT

reusing existing storage account: staidemosclsfrdev01


## 3. Grant the uploader write access

The upload below uses `DefaultAzureCredential`. In this repo that resolves to the
workshop service principal (`AZURE_CLIENT_ID`), which has Cognitive Services access
but **no storage data role** by default — so grant it `Storage Blob Data Contributor`
first. Role id `ba92f5b4-2d11-453d-a403-e96b0029c9fe` is that role.

In [9]:
STORAGE_BLOB_DATA_CONTRIBUTOR = 'ba92f5b4-2d11-453d-a403-e96b0029c9fe'

_client_id = os.environ.get('AZURE_CLIENT_ID', '')
uploader_oid = az(f'ad sp show --id {_client_id} --query id -o tsv', quiet=True) if _client_id else None
_acct_id = az(f'storage account show -n {ACCOUNT} -g {RESOURCE_GROUP} --query id -o tsv', quiet=True)

if uploader_oid and _acct_id:
    az(f'role assignment create --assignee-object-id {uploader_oid} '
       f'--assignee-principal-type ServicePrincipal '
       f'--role {STORAGE_BLOB_DATA_CONTRIBUTOR} --scope {_acct_id}', quiet=True)
    print('uploader granted Storage Blob Data Contributor (or already present).')
else:
    print('No AZURE_CLIENT_ID in env — the uploader is your az-login identity.')
    print('Ensure it has Storage Blob Data Contributor on', ACCOUNT)

uploader granted Storage Blob Data Contributor (or already present).


## 4. Upload the training data

Each class folder is uploaded under its own prefix: `invoice/…` and `claim/…`.
The prefix name is what becomes the classifier's `docType`. The retry absorbs the
brief delay before the role assignment above takes effect.

In [10]:
import time

counts = None
for attempt in range(1, 5):
    try:
        counts = _lib.upload_training_data(account_name=ACCOUNT)
        break
    except Exception as e:  # data-plane RBAC can take ~30s to propagate
        if attempt == 4:
            raise
        print(f'  attempt {attempt} failed ({type(e).__name__}); waiting for RBAC to propagate ...')
        time.sleep(20)

print()
for label, n in counts.items():
    status = 'ok' if n >= 5 else 'TOO FEW (need >= 5)'
    print(f'  {label:8s}: {n} files  [{status}]')
assert all(n >= 5 for n in counts.values()), 'Each class needs at least 5 training samples.'

 created container 'classifier-training'
  invoice  -> 5 file(s) under 'invoice/'
  claim    -> 8 file(s) under 'claim/'

  invoice : 5 files  [ok]
  claim   : 8 files  [ok]


## 5. Grant the classifier read access (managed identity)

The Foundry resource's system-assigned identity needs `Storage Blob Data Reader`
on the account so training in notebook `02` can read the container. Role id
`2a2b9908-6ea1-4ae2-8e65-a410df84e7d1` is **Storage Blob Data Reader**.

In [11]:
FOUNDRY_NAME = os.environ.get('FOUNDRY_RESOURCE_NAME', '')
STORAGE_BLOB_DATA_READER = '2a2b9908-6ea1-4ae2-8e65-a410df84e7d1'

principal_id = None
if FOUNDRY_NAME:
    principal_id = az(f'cognitiveservices account show -n {FOUNDRY_NAME} -g {RESOURCE_GROUP} '
                      f'--query identity.principalId -o tsv', quiet=True)
account_id = az(f'storage account show -n {ACCOUNT} -g {RESOURCE_GROUP} --query id -o tsv', quiet=True)

if principal_id and account_id:
    print('foundry MSI principalId:', principal_id)
    az(f'role assignment create --assignee-object-id {principal_id} '
       f'--assignee-principal-type ServicePrincipal '
       f'--role {STORAGE_BLOB_DATA_READER} --scope {account_id}', quiet=True)
    print('Storage Blob Data Reader assigned to the Foundry identity (or already present).')
else:
    print('Could not resolve the Foundry MSI or storage id.')
    print('Set FOUNDRY_RESOURCE_NAME + AZURE_RESOURCE_GROUP, or assign')
    print('Storage Blob Data Reader to the Foundry identity on', ACCOUNT, 'manually.')

foundry MSI principalId: 54376e82-9f43-46db-8961-67dfe7ef6cc6
Storage Blob Data Reader assigned to the Foundry identity (or already present).


## 6. Verify the upload

List the blobs grouped by class prefix.

In [12]:
svc = _lib.get_blob_service_client(ACCOUNT)
cc = svc.get_container_client(_lib.CONTAINER)
by_prefix = {}
for b in cc.list_blobs():
    by_prefix.setdefault(b.name.split('/')[0], []).append(b.name)
for prefix, names in sorted(by_prefix.items()):
    print(f'{prefix}/  ({len(names)} blobs)')
    for n in names:
        print('   ', n)

claim/  (8 blobs)
    claim/ClaimForm_1.pdf
    claim/ClaimForm_2.pdf
    claim/ClaimForm_3.pdf
    claim/ClaimForm_4.pdf
    claim/ClaimForm_5.pdf
    claim/ClaimForm_6.pdf
    claim/ClaimForm_7.pdf
    claim/claimform_handwritten_1.pdf
invoice/  (5 blobs)
    invoice/FabrikamInvoice_1.pdf
    invoice/FabrikamInvoice_2.pdf
    invoice/FabrikamInvoice_3.pdf
    invoice/FabrikamInvoice_4.pdf
    invoice/FabrikamInvoice_5.pdf


Container URL the build will use (no SAS — managed identity handles access):

In [13]:
print(_lib.container_url(ACCOUNT))
print('\nNext: run 02_train_classifier.ipynb')

https://staidemosclsfrdev01.blob.core.windows.net/classifier-training

Next: run 02_train_classifier.ipynb
